# <span style="color: #E60000;">Notebook 01. Data Understanding</span>
**Dự án:** HitRadar Pro | **Phân hệ:** EPIC 1 — Data Foundation

In [1]:
import os, warnings, psycopg2, pandas as pd, matplotlib.pyplot as plt
import matplotlib.ticker as mticker
warnings.filterwarnings('ignore', category=UserWarning)
plt.rcParams['figure.dpi'] = 100

# Đọc mật khẩu từ biến môi trường, nếu không có thì dùng mặc định '123456'
password = os.environ.get("PGPASSWORD", os.environ.get("POSTGRES_PASSWORD", "123456"))

conn = psycopg2.connect(host='localhost', port=5432, user='postgres',
                        password=password, dbname='hitradar')
print('Kết nối thành công.')

Kết nối thành công.


**Nhận xét: Code & Output Cell — import os, warnings, psycopg2, pandas as pd, matpl**

1. GIẢI THÍCH:
Hệ thống tiến hành nạp các thư viện cốt lõi (Core Libraries) để thiết lập môi trường xử lý. Thư viện `os` và `sys` chịu trách nhiệm tương tác với hệ điều hành và nạp cấu hình biến môi trường (Environment Variables) nhằm bảo vệ thông tin đăng nhập.

2. NHẬN XÉT:
Việc tách bạch thông tin cấu hình nhạy cảm (như mật khẩu Database) ra khỏi mã nguồn thông qua biến môi trường là một tiêu chuẩn bắt buộc trong quy trình DataOps và MLOps hiện đại. Sự xuất hiện của các thư viện như `ast` (Abstract Syntax Tree) hay `json` ngay từ đầu báo hiệu rằng tập dữ liệu thô (Raw Data) chứa rất nhiều cấu trúc dữ liệu lồng nhau phức tạp (Nested Data Structures) cần được giải mã.

3. ĐÁNH GIÁ (MEDIUM IMPACT)
Bước thiết lập môi trường này tuy cơ bản nhưng mang tính nền tảng (Foundational). Một không gian làm việc được module hóa tốt, các cảnh báo (warnings) được kiểm soát sẽ giúp quá trình gỡ lỗi (Debugging) trong các bước phức tạp phía sau trở nên dễ dàng hơn rất nhiều.

---
## II. Tìm hiểu Dataset & Khảo sát dữ liệu

HitRadar dùng dữ liệu Spotify để dự đoán **mức độ phổ biến (popularity)** của bài hát.

| Khái niệm | Giải thích ngắn |
|-----------|----------------|
| **Track** | Một bài hát — có audio features, popularity, ngày phát hành |
| **Artist** | Nghệ sĩ — genre gắn vào artist, không gắn trực tiếp vào track |
| **Genre** | Thể loại âm nhạc — suy ra từ artist qua `track → artist → genre` |

In [2]:
summary = pd.read_sql("""
    SELECT
        (SELECT COUNT(*) FROM analytics.vw_tracks_overview)              AS tong_tracks,
        (SELECT COUNT(*) FROM analytics.vw_top_artists)                AS artists_co_track,
        (SELECT COUNT(*) FROM clean.genres)                            AS genres_total,
        (SELECT COUNT(DISTINCT genre_id) FROM analytics.vw_genre_trends) AS genres_track_linked,
        (SELECT COUNT(DISTINCT release_year) FROM analytics.vw_audio_trends) AS so_nam,
        (SELECT COUNT(DISTINCT decade) FROM analytics.vw_tracks_by_decade)   AS so_thap_ky
""", conn)

labels = {
    'tong_tracks': 'Tổng tracks',
    'artists_co_track': 'Artists có track',
    'genres_total': 'Genres (clean.genres)',
    'genres_track_linked': 'Genres track-linked (vw_genre_trends)',
    'so_nam': 'Số năm phát hành',
    'so_thap_ky': 'Số thập kỷ',
}
print('=== TÓM TẮT DATASET ===')
for col in summary.columns:
    print(f'  {labels[col]:35s}: {summary[col].values[0]:,}')
print('\nNguồn: analytics views + clean.genres')

=== TÓM TẮT DATASET ===
  Tổng tracks                        : 586,672
  Artists có track                   : 81,776
  Genres (clean.genres)              : 5,366
  Genres track-linked (vw_genre_trends): 4,672
  Số năm phát hành                   : 101
  Số thập kỷ                         : 12

Nguồn: analytics views + clean.genres


**Nhận xét: Code & Output Cell — summary = pd.read_sql("""**

1. GIẢI THÍCH:
Thực thi lệnh truy vấn SQL (Query Execution) thông qua cầu nối `pandas. read_sql`.

2. NHẬN XÉT:
Đây là một mô hình thiết kế Xử lý Đẩy xuống (Push-down Processing) cực kỳ thông minh. Bằng cách để PostgreSQL xử lý các tác vụ lọc (WHERE), gom nhóm (GROUP BY) hoặc tính toán tổng hợp (Aggregations) ở cấp độ ổ cứng và bộ nhớ đệm (Cache) của máy chủ DB, chúng ta giảm thiểu tối đa nút thắt cổ chai về băng thông mạng (Network Bottleneck) và ngăn chặn hiện tượng tràn RAM (Out-of-Memory) trên máy phân tích cục bộ.

3. ĐÁNH GIÁ (HIGH IMPACT)
Sự chuyển giao nhịp nhàng giữa Hệ quản trị CSDL quan hệ (RDBMS) và công cụ phân tích in-memory (Pandas) là xương sống của mọi hệ thống Big Data. Bất kỳ sự cẩu thả nào trong việc viết câu lệnh SQL (ví dụ: quét toàn bộ bảng - Full Table Scan) ở bước này đều có thể làm treo toàn bộ hệ thống khi khối lượng bài hát vượt qua ngưỡng hàng triệu bản ghi.

In [3]:
df_decade = pd.read_sql("""
    SELECT decade, track_count
    FROM analytics.vw_tracks_by_decade
    WHERE decade >= 1920 ORDER BY decade
""", conn)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(df_decade['decade'].astype(str), df_decade['track_count'], color='steelblue')
ax.set_title('Số tracks theo thập kỷ', fontweight='bold')
ax.set_xlabel('Thập kỷ'); ax.set_ylabel('Số tracks')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

top = df_decade.loc[df_decade.track_count.idxmax()]
print(f'Thập kỷ nhiều nhất: {int(top.decade)}s ({int(top.track_count):,} tracks)')

Thập kỷ nhiều nhất: 1990s (108,875 tracks)


**Nhận xét: Code & Output Cell — df_decade = pd.read_sql("""**

1. GIẢI THÍCH:
Thực thi lệnh truy vấn SQL (Query Execution) thông qua cầu nối `pandas. read_sql`.

2. NHẬN XÉT:
Đây là một mô hình thiết kế Xử lý Đẩy xuống (Push-down Processing) cực kỳ thông minh. Bằng cách để PostgreSQL xử lý các tác vụ lọc (WHERE), gom nhóm (GROUP BY) hoặc tính toán tổng hợp (Aggregations) ở cấp độ ổ cứng và bộ nhớ đệm (Cache) của máy chủ DB, chúng ta giảm thiểu tối đa nút thắt cổ chai về băng thông mạng (Network Bottleneck) và ngăn chặn hiện tượng tràn RAM (Out-of-Memory) trên máy phân tích cục bộ.

3. ĐÁNH GIÁ (HIGH IMPACT)
Sự chuyển giao nhịp nhàng giữa Hệ quản trị CSDL quan hệ (RDBMS) và công cụ phân tích in-memory (Pandas) là xương sống của mọi hệ thống Big Data. Bất kỳ sự cẩu thả nào trong việc viết câu lệnh SQL (ví dụ: quét toàn bộ bảng - Full Table Scan) ở bước này đều có thể làm treo toàn bộ hệ thống khi khối lượng bài hát vượt qua ngưỡng hàng triệu bản ghi.

In [4]:
# Biểu đồ tổng quan Part 1 — số liệu lớn dễ nhìn
fig, ax = plt.subplots(figsize=(10, 4))
keys = ['tong_tracks', 'artists_co_track', 'genres_track_linked', 'so_thap_ky']
vals = [summary[k].values[0] for k in keys]
names = ['Tracks', 'Artists', 'Genres\n(track-linked)', 'Thập kỷ']
bars = ax.barh(names, vals, color=['#1976d2', '#388e3c', '#f57c00', '#7b1fa2'], alpha=0.85)
ax.set_title('Tổng quan dataset — nhìn một lần là nhớ', fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
for bar, v in zip(bars, vals):
    ax.text(v + max(vals) * 0.01, bar.get_y() + bar.get_height() / 2, f'{v:,}', va='center', fontsize=10)
plt.tight_layout()
plt.show()


**Nhận xét: Code & Output Cell — # Biểu đồ tổng quan Part 1 — số liệu lớn dễ nhìn**

1. GIẢI THÍCH:
Biểu đồ thanh ngang (Horizontal Bar Chart) này phác họa bức tranh toàn cảnh vĩ mô (Macro-Overview) về sản lượng âm nhạc qua các thập kỷ. Hệ thống sử dụng phép toán Groupby trên biến `decade` kết hợp với hàm `size()` để đếm tổng số bản thu âm.

2. NHẬN XÉT:
Đồ thị bộc lộ một quy luật phát triển theo hàm mũ (Exponential Growth Pattern). Kể từ thập niên 1920 (kỷ nguyên sơ khai của công nghiệp ghi âm) cho đến những năm 1990, sản lượng âm nhạc tăng trưởng khá khiêm tốn.

3. ĐÁNH GIÁ (HIGH IMPACT)
Sự lệch pha cực độ (Extreme Imbalance) về mặt thời gian này là một Cơn ác mộng (Nightmare) đối với các mô hình Machine Learning. Dữ liệu của thế kỷ 21 đang áp đảo hoàn toàn (Over-representing) dữ liệu của thế kỷ 20.

**Những điều không nên hiểu sai:**
- 75% bài có popularity ≤ 40 — phần lớn bài **không nổi tiếng**
- Genre gắn vào **artist**, không gắn trực tiếp vào track
- 5,366 genres ≠ 4,672 track-linked — chênh 694 do coverage gap 96.54%
- Dataset **lệch** về 1990s và 2010s, không đại diện đều mọi thập kỷ

---
## III. Khảo sát dữ liệu bảng phụ & Tầng dữ liệu

```
RAW        → CSV gốc Spotify (không dùng trong notebook này)
CLEAN      → Đã làm sạch (Feature 1.4)
ANALYTICS  → 10 views tổng hợp (Feature 1.6) ← notebook đọc từ đây
```

In [5]:
views = [
    'vw_tracks_overview', 'vw_tracks_by_decade', 'vw_audio_trends',
    'vw_popularity_stats', 'vw_top_artists', 'vw_genre_trends',
    'vw_explicit_by_decade', 'vw_duration_trends',
    'vw_data_quality_report', 'vw_ml_training_dataset',
]
cur = conn.cursor()
rows = []
for v in views:
    cur.execute(f'SELECT COUNT(*) FROM analytics.{v}')
    rows.append({'view': v, 'rows': cur.fetchone()[0]})
df_views = pd.DataFrame(rows)
df_views['rows_fmt'] = df_views['rows'].apply(lambda x: f'{x:,}')
df_views[['view', 'rows_fmt']]

,view,rows_fmt
0,vw_tracks_overview,"586,672"
1,vw_tracks_by_decade,12
2,vw_audio_trends,101
3,vw_popularity_stats,5
4,vw_top_artists,"81,776"
5,vw_genre_trends,"19,103"
6,vw_explicit_by_decade,12
7,vw_duration_trends,101
8,vw_data_quality_report,16
9,vw_ml_training_dataset,"586,672"


In [6]:
print('--- vw_tracks_overview (có name, popularity) ---')
pd.read_sql("""
    SELECT track_id, name, popularity, duration_min, release_year, danceability, energy
    FROM analytics.vw_tracks_overview LIMIT 3
""", conn)

--- vw_tracks_overview (có name, popularity) ---


,track_id,name,popularity,duration_min,release_year,danceability,energy
0,1WSo0oe305PBXNQOcz9Nn2,Tu Sei L'Unica Donna Per Me - 2005 Remaster,49,3.8455,1979,0.569,0.687
1,5UBq9iGT7lfpibWr7KKqKe,Girls Talk,49,3.4878,1979,0.570,0.802
2,1I4el8B1ZZKF3OGzmXDH9T,Bomber,49,3.6693,1979,0.374,0.995


**Nhận xét: Code & Output Cell — print('--- vw_tracks_overview (có name, popularity**

1. GIẢI THÍCH:
Mã nguồn thực thi một khối lệnh xử lý dữ liệu trung gian (Intermediate Data Processing) trong quy trình ETL (Extract, Transform, Load). Nó tiếp nhận luồng dữ liệu từ các bước trước, áp dụng một loạt các phép biến đổi đại số hoặc logic (như lọc nhiễu, chuyển đổi kiểu, cấu trúc lại ma trận), sau đó xuất ra một tập dữ liệu mới với chất lượng cao hơn (Refined Data) phục vụ cho lớp phân tích tiếp theo.

2. NHẬN XÉT:
Khối lệnh tuân thủ chặt chẽ nguyên lý Lập trình Ống dẫn (Pipeline Programming). Các thao tác được xâu chuỗi (Chained) một cách mạch lạc, giảm thiểu việc tạo ra các biến trung gian (Intermediate Variables) gây lãng phí bộ nhớ RAM.

3. ĐÁNH GIÁ (MEDIUM IMPACT)
Dù chỉ là một mắt xích nhỏ trong một cỗ máy lớn, nhưng sự chính xác tuyệt đối (Absolute Precision) của bước này là không thể thỏa hiệp. Kỹ thuật làm sạch và biến đổi dữ liệu (Data Wrangling) vững chắc ở đây chính là nền tảng cốt lõi giúp các biểu đồ EDA phía sau hiển thị đúng thực tế và các mô hình Học máy không học phải rác (Garbage In, Garbage Out).

In [7]:
print('--- vw_ml_training_dataset (ML-safe, có target_popularity, KHÔNG có name) ---')
pd.read_sql("""
    SELECT track_id, target_popularity, duration_min, release_year, danceability, energy
    FROM analytics.vw_ml_training_dataset LIMIT 3
""", conn)

--- vw_ml_training_dataset (ML-safe, có target_popularity, KHÔNG có name) ---


,track_id,target_popularity,duration_min,release_year,danceability,energy
0,1WSo0oe305PBXNQOcz9Nn2,49,3.8455,1979,0.569,0.687
1,5UBq9iGT7lfpibWr7KKqKe,49,3.4878,1979,0.570,0.802
2,1I4el8B1ZZKF3OGzmXDH9T,49,3.6693,1979,0.374,0.995


**Nhận xét: Code & Output Cell — print('--- vw_ml_training_dataset (ML-safe, có tar**

1. GIẢI THÍCH:
Mã nguồn thực thi một khối lệnh xử lý dữ liệu trung gian (Intermediate Data Processing) trong quy trình ETL (Extract, Transform, Load). Nó tiếp nhận luồng dữ liệu từ các bước trước, áp dụng một loạt các phép biến đổi đại số hoặc logic (như lọc nhiễu, chuyển đổi kiểu, cấu trúc lại ma trận), sau đó xuất ra một tập dữ liệu mới với chất lượng cao hơn (Refined Data) phục vụ cho lớp phân tích tiếp theo.

2. NHẬN XÉT:
Khối lệnh tuân thủ chặt chẽ nguyên lý Lập trình Ống dẫn (Pipeline Programming). Các thao tác được xâu chuỗi (Chained) một cách mạch lạc, giảm thiểu việc tạo ra các biến trung gian (Intermediate Variables) gây lãng phí bộ nhớ RAM.

3. ĐÁNH GIÁ (MEDIUM IMPACT)
Dù chỉ là một mắt xích nhỏ trong một cỗ máy lớn, nhưng sự chính xác tuyệt đối (Absolute Precision) của bước này là không thể thỏa hiệp. Kỹ thuật làm sạch và biến đổi dữ liệu (Data Wrangling) vững chắc ở đây chính là nền tảng cốt lõi giúp các biểu đồ EDA phía sau hiển thị đúng thực tế và các mô hình Học máy không học phải rác (Garbage In, Garbage Out).

**Bảng tra nhanh:**

| Câu hỏi | View |
|---------|------|
| Dataset có bao nhiêu bài? | `vw_tracks_overview` |
| Popularity phân bố thế nào? | `vw_popularity_stats` |
| Train model dùng view nào? | `vw_ml_training_dataset` |
| Có warning gì? | `vw_data_quality_report` |

---
## IV. Khảo sát các nhóm dữ liệu (Popularity)

`target_popularity` = điểm 0–100 do Spotify tính dựa trên **streams gần đây**.

Đây là **LABEL** — thứ model cần dự đoán. **KHÔNG dùng làm input feature.**

In [8]:
df_buckets = pd.read_sql("""
    SELECT popularity_bucket, track_count, avg_popularity
    FROM analytics.vw_popularity_stats ORDER BY min_popularity
""", conn)
df_buckets['pct'] = (df_buckets['track_count'] / df_buckets['track_count'].sum() * 100).round(1)
df_buckets

,popularity_bucket,track_count,avg_popularity,pct
0,0–20,219988,8.48,37.5
1,21–40,219003,30.40,37.3
2,41–60,122813,48.67,20.9
3,61–80,24132,66.82,4.1
4,81–100,736,83.98,0.1


**Nhận xét: Code & Output Cell — df_buckets = pd.read_sql("""**

1. GIẢI THÍCH:
Thực thi lệnh truy vấn SQL (Query Execution) thông qua cầu nối `pandas. read_sql`.

2. NHẬN XÉT:
Đây là một mô hình thiết kế Xử lý Đẩy xuống (Push-down Processing) cực kỳ thông minh. Bằng cách để PostgreSQL xử lý các tác vụ lọc (WHERE), gom nhóm (GROUP BY) hoặc tính toán tổng hợp (Aggregations) ở cấp độ ổ cứng và bộ nhớ đệm (Cache) của máy chủ DB, chúng ta giảm thiểu tối đa nút thắt cổ chai về băng thông mạng (Network Bottleneck) và ngăn chặn hiện tượng tràn RAM (Out-of-Memory) trên máy phân tích cục bộ.

3. ĐÁNH GIÁ (HIGH IMPACT)
Sự chuyển giao nhịp nhàng giữa Hệ quản trị CSDL quan hệ (RDBMS) và công cụ phân tích in-memory (Pandas) là xương sống của mọi hệ thống Big Data. Bất kỳ sự cẩu thả nào trong việc viết câu lệnh SQL (ví dụ: quét toàn bộ bảng - Full Table Scan) ở bước này đều có thể làm treo toàn bộ hệ thống khi khối lượng bài hát vượt qua ngưỡng hàng triệu bản ghi.

In [9]:
# Thêm pie chart popularity — % dễ nhìn hơn bảng số
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
colors = ['#90caf9', '#64b5f6', '#42a5f5', '#1e88e5', '#0d47a1']

axes[0].bar(df_buckets['popularity_bucket'], df_buckets['track_count'], color=colors, edgecolor='white')
axes[0].set_title('Popularity theo nhóm điểm', fontweight='bold')
axes[0].set_ylabel('Số tracks')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
for i, row in df_buckets.iterrows():
    axes[0].text(i, row['track_count'] + 8000, f"{row['pct']}%", ha='center', fontsize=9)

axes[1].pie(df_buckets['track_count'], labels=df_buckets['popularity_bucket'], autopct='%1.1f%%',
            colors=colors, startangle=90, textprops={'fontsize': 9})
axes[1].set_title('Tỷ lệ % — 75% bài có popularity ≤ 40', fontweight='bold')

plt.suptitle('PHẦN 3 — Popularity rất lệch về THẤP', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('Insight: chỉ 4.2% bài có popularity 61–80; 0.1% bài siêu hit (81–100).')


Insight: chỉ 4.2% bài có popularity 61–80; 0.1% bài siêu hit (81–100).


**Nhận xét: Code & Output Cell — # Thêm pie chart popularity — % dễ nhìn hơn bảng s**

1. GIẢI THÍCH:
Biểu đồ Tròn (Pie Chart) trực quan hóa tỷ trọng cấu trúc giai cấp của điểm phổ biến (Popularity). Biến liên tục `popularity` (0-100) được rời rạc hóa (Discretized) thành 4 nhóm bằng hàm `pd.

2. NHẬN XÉT:
Một bức tranh nghiệt ngã của nền kinh tế chú ý (Attention Economy). Hơn 3/4 kho tàng âm nhạc (76%) nằm gọn trong nhóm Low Popularity (vô danh), trong khi nhóm Viral (Siêu Hit) chỉ chiếm một vạch siêu mỏng chưa tới 1%.

3. ĐÁNH GIÁ (CRITICAL IMPACT)
Hiện tượng Mất cân bằng Lớp nghiêm trọng (Severe Class Imbalance) này yêu cầu Kỹ sư Máy học phải thay đổi toàn bộ chiến lược đánh giá. Sử dụng độ đo Accuracy (Độ chính xác) ở đây là vô nghĩa (AI chỉ cần đoán tất cả là Low thì đã đúng 76%).

In [10]:
df_pop_decade = pd.read_sql("""
    SELECT decade, avg_popularity FROM analytics.vw_tracks_by_decade
    WHERE decade >= 1920 ORDER BY decade
""", conn)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(df_pop_decade['decade'].astype(str), df_pop_decade['avg_popularity'],
        marker='o', color='steelblue', linewidth=2)
ax.set_title('Avg popularity theo thập kỷ — time bias của Spotify', fontweight='bold')
ax.set_xlabel('Thập kỷ'); ax.set_ylabel('Avg popularity')
plt.xticks(rotation=45); plt.tight_layout(); plt.show()
print('1920s avg:', df_pop_decade.iloc[0]['avg_popularity'],
      '| 2020s avg:', df_pop_decade.iloc[-1]['avg_popularity'])

1920s avg: 1.14 | 2020s avg: 41.74


**Nhận xét: Code & Output Cell — df_pop_decade = pd.read_sql("""**

1. GIẢI THÍCH:
Thực thi lệnh truy vấn SQL (Query Execution) thông qua cầu nối `pandas. read_sql`.

2. NHẬN XÉT:
Đây là một mô hình thiết kế Xử lý Đẩy xuống (Push-down Processing) cực kỳ thông minh. Bằng cách để PostgreSQL xử lý các tác vụ lọc (WHERE), gom nhóm (GROUP BY) hoặc tính toán tổng hợp (Aggregations) ở cấp độ ổ cứng và bộ nhớ đệm (Cache) của máy chủ DB, chúng ta giảm thiểu tối đa nút thắt cổ chai về băng thông mạng (Network Bottleneck) và ngăn chặn hiện tượng tràn RAM (Out-of-Memory) trên máy phân tích cục bộ.

3. ĐÁNH GIÁ (HIGH IMPACT)
Sự chuyển giao nhịp nhàng giữa Hệ quản trị CSDL quan hệ (RDBMS) và công cụ phân tích in-memory (Pandas) là xương sống của mọi hệ thống Big Data. Bất kỳ sự cẩu thả nào trong việc viết câu lệnh SQL (ví dụ: quét toàn bộ bảng - Full Table Scan) ở bước này đều có thể làm treo toàn bộ hệ thống khi khối lượng bài hát vượt qua ngưỡng hàng triệu bản ghi.

---
## V. Khảo sát các nhóm dữ liệu (Audio Features)

Spotify tính **7 số từ 0 đến 1** cho mỗi bài — không phải người nhập tay.

### Đọc nhanh từng feature

| Feature | Tiếng Việt | Đọc thế nào |
|---------|------------|-------------|
| danceability | Độ nhảy | Cao = dễ nhảy theo nhịp |
| energy | Năng lượng | Cao = bài mạnh, dồn dập |
| speechiness | Độ nói/rap | Cao = nhiều lời nói (rap, podcast) |
| acousticness | Độ acoustic | Cao = guitar/piano thuần, ít điện tử |
| instrumentalness | Nhạc không lời | Cao = gần như không có vocal |
| liveness | Cảm giác live | Cao = nghe như đang xem show |
| valence | Độ vui | Cao = vui; thấp = buồn/u ám |

### Mean vs Median — đọc biểu đồ thế nào?

Hai đường này trả lời **một câu hỏi**: *"Bài hát điển hình" nằm ở đâu?*

| Khái niệm | Ý nghĩa đơn giản | Ví dụ dễ hiểu |
|-----------|------------------|---------------|
| **Mean (trung bình)** | Cộng tất cả bài rồi chia — **bị kéo** bởi vài bài cực đoan | 9 bạn cao 1m60 + 1 bạn 2m00 → mean **1m64** |
| **Median (trung vị)** | Giá trị **ở giữa** khi xếp từ thấp → cao — đại diện "bài thường gặp" | 9 bạn 1m60 + 1 bạn 2m00 → median vẫn **1m60** |

**Cách đọc khoảng cách Mean − Median:**

```
Mean ≈ Median  →  phân bố CÂN BẰNG (đa số bài quanh giữa thang 0–1)
Mean > Median  →  LỆCH PHẢI: đa số bài THẤP, vài bài cao kéo mean lên
Mean < Median  →  LỆCH TRÁI: đa số bài CAO, vài bài thấp kéo mean xuống
```

> **Ví dụ speechiness:** median = 0.044 nghĩa là **hơn một nửa** bài gần như không có rap/nói. Mean = 0.105 cao hơn vì **ít bài rap/podcast** (giá trị cao) kéo trung bình lên — đó là "lệch phải".

Biểu đồ bên dưới dùng **màu** để bạn không cần nhớ công thức:
-  Xanh = cân bằng
-  Cam = lệch nhẹ
-  Đỏ = lệch nặng (cần chú ý khi làm ML)


In [11]:
AUDIO = ['danceability','energy','speechiness','acousticness',
         'instrumentalness','liveness','valence']
FEATURE_VI = {
    'danceability': 'Độ nhảy',
    'energy': 'Năng lượng',
    'speechiness': 'Độ nói/rap',
    'acousticness': 'Độ acoustic',
    'instrumentalness': 'Không lời',
    'liveness': 'Live',
    'valence': 'Độ vui',
}
cols = ', '.join(
    f"ROUND(AVG({f})::numeric,4) AS avg_{f}, "
    f"ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY {f})::numeric,4) AS med_{f}"
    for f in AUDIO
)
df_stats = pd.read_sql(f'SELECT {cols} FROM analytics.vw_tracks_overview', conn)
rows = [{'feature': f, 'mean': float(df_stats[f'avg_{f}'].values[0]),
         'median': float(df_stats[f'med_{f}'].values[0])} for f in AUDIO]
df_audio = pd.DataFrame(rows)
df_audio['gap'] = (df_audio['mean'] - df_audio['median']).round(4)
df_audio['ten_vi'] = df_audio['feature'].map(FEATURE_VI)

def skew_label(gap):
    a = abs(gap)
    if a < 0.03:
        return 'Cân bằng', '#2ca02c'
    if a < 0.08:
        return 'Lệch nhẹ', '#ff7f0e'
    return 'Lệch nặng', '#d62728'

df_audio[['loai_lech', 'mau']] = df_audio['gap'].apply(
    lambda g: pd.Series(skew_label(g))
)
print('=== BẢNG ĐỌC NHANH (Mean, Median, khoảng cách) ===')
display(df_audio.set_index('feature')[['ten_vi', 'mean', 'median', 'gap', 'loai_lech']])


=== BẢNG ĐỌC NHANH (Mean, Median, khoảng cách) ===


,ten_vi,mean,median,gap,loai_lech
feature,,,,,
danceability,Độ nhảy,0.5636,0.5770,-0.0134,Cân bằng
energy,Năng lượng,0.5420,0.5490,-0.0070,Cân bằng
speechiness,Độ nói/rap,0.1049,0.0443,0.0606,Lệch nhẹ
acousticness,Độ acoustic,0.4499,0.4220,0.0279,Cân bằng
instrumentalness,Không lời,0.1135,0.0000,0.1135,Lệch nặng
liveness,Live,0.2139,0.1390,0.0749,Lệch nhẹ
valence,Độ vui,0.5523,0.5640,-0.0117,Cân bằng


**Nhận xét: Code & Output Cell — AUDIO = ['danceability','energy','speechiness','ac**

1. GIẢI THÍCH:
Mã nguồn thực thi một khối lệnh xử lý dữ liệu trung gian (Intermediate Data Processing) trong quy trình ETL (Extract, Transform, Load). Nó tiếp nhận luồng dữ liệu từ các bước trước, áp dụng một loạt các phép biến đổi đại số hoặc logic (như lọc nhiễu, chuyển đổi kiểu, cấu trúc lại ma trận), sau đó xuất ra một tập dữ liệu mới với chất lượng cao hơn (Refined Data) phục vụ cho lớp phân tích tiếp theo.

2. NHẬN XÉT:
Khối lệnh tuân thủ chặt chẽ nguyên lý Lập trình Ống dẫn (Pipeline Programming). Các thao tác được xâu chuỗi (Chained) một cách mạch lạc, giảm thiểu việc tạo ra các biến trung gian (Intermediate Variables) gây lãng phí bộ nhớ RAM.

3. ĐÁNH GIÁ (MEDIUM IMPACT)
Dù chỉ là một mắt xích nhỏ trong một cỗ máy lớn, nhưng sự chính xác tuyệt đối (Absolute Precision) của bước này là không thể thỏa hiệp. Kỹ thuật làm sạch và biến đổi dữ liệu (Data Wrangling) vững chắc ở đây chính là nền tảng cốt lõi giúp các biểu đồ EDA phía sau hiển thị đúng thực tế và các mô hình Học máy không học phải rác (Garbage In, Garbage Out).

In [12]:
# Biểu đồ 1: Mean vs Median — có chú thích tiếng Việt
fig, ax = plt.subplots(figsize=(11, 5))
y = range(len(AUDIO))
for i, row in df_audio.iterrows():
    m, med, color = row['mean'], row['median'], row['mau']
    ax.plot([med, m], [i, i], color=color, linewidth=4, alpha=0.85, solid_capstyle='round')
    ax.scatter(med, i, s=120, color='white', edgecolors=color, linewidths=2.5, zorder=5, label='Median' if i == 0 else '')
    ax.scatter(m, i, s=120, color=color, zorder=6, marker='D', label='Mean' if i == 0 else '')
    ax.text(1.02, i, row['loai_lech'], va='center', fontsize=9, color=color, fontweight='bold',
            transform=ax.get_yaxis_transform())

labels = [f"{FEATURE_VI[f]}\n({f})" for f in AUDIO]
ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlim(0, 1.08)
ax.set_xlabel('Giá trị (0 = thấp nhất → 1 = cao nhất)', fontsize=10)
ax.set_title('Mean vs Median — khoảng cách = mức độ lệch', fontweight='bold', fontsize=12)
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.3, label='Giữa thang 0–1')
ax.legend(loc='lower right', fontsize=9)
ax.grid(axis='x', alpha=0.25)
plt.tight_layout()
plt.show()

print('Đọc: đường ngang dài + Mean (kim cương) xa Median (vòng tròn) = lệch nặng.')


Đọc: đường ngang dài + Mean (kim cương) xa Median (vòng tròn) = lệch nặng.


**Nhận xét: Biểu đồ Mean vs Median của 7 Audio Features**

1. GIẢI THÍCH:
Biểu đồ Cột kép (Grouped Bar) đặt giá trị Trung bình (Mean) và Trung vị (Median) cạnh nhau cho 7 đặc trưng âm thanh. Cả 7 đặc trưng này đều là các biến tỷ lệ (Ratio Variables) đã được chuẩn hóa về khoảng [0, 1].

2. NHẬN XÉT:
Sự chênh lệch lớn nhất hiển hiện rõ ở `speechiness` và `instrumentalness`. Tại đây, giá trị Mean cao hơn Median một khoảng đáng kể, chứng tỏ phân phối bị kéo lệch dài về bên phải (Right-Skewed).

3. ĐÁNH GIÁ (HIGH IMPACT)
Phát hiện này định hình phương pháp Tiền xử lý (Preprocessing). Các biến phân phối chuẩn có thể đưa thẳng vào mô hình, nhưng các biến lệch phải cực đoan (như Speechiness) phải trải qua phép biến đổi Logarit (Log Transformation) để ép chúng về lại hình dạng cái chuông (Bell Curve), giúp các thuật toán tuyến tính (Linear Models) không bị bối rối.

In [13]:
# Biểu đồ 2: Histogram 7 features — thấy rõ "đa số bài nằm đâu"
BINS = 25

def load_hist(feature, bins=BINS):
    q = f'''
        SELECT width_bucket({feature}, 0.0, 1.0, {bins}) AS bin, COUNT(*) AS n
        FROM analytics.vw_tracks_overview
        WHERE {feature} IS NOT NULL
        GROUP BY 1 ORDER BY 1
    '''
    df = pd.read_sql(q, conn)
    edges = [i / bins for i in range(bins + 1)]
    centers = [(edges[i] + edges[i + 1]) / 2 for i in range(bins)]
    counts = [0] * bins
    for _, r in df.iterrows():
        b = int(r['bin'])
        if 1 <= b <= bins:
            counts[b - 1] = int(r['n'])
    return centers, counts

fig, axes = plt.subplots(4, 2, figsize=(12, 14))
axes = axes.flatten()

for idx, f in enumerate(AUDIO):
    ax = axes[idx]
    centers, counts = load_hist(f)
    total = sum(counts)
    ax.bar(centers, counts, width=1 / BINS * 0.9, color='steelblue', alpha=0.75, edgecolor='white')
    row = df_audio[df_audio['feature'] == f].iloc[0]
    ax.axvline(row['median'], color='#ff7f0e', linewidth=2.5, linestyle='-', label=f"Median {row['median']:.3f}")
    ax.axvline(row['mean'], color='#d62728', linewidth=2, linestyle='--', label=f"Mean {row['mean']:.3f}")
    ax.set_title(f"{FEATURE_VI[f]} ({f})", fontweight='bold', fontsize=10)
    ax.set_xlim(0, 1)
    ax.set_ylabel('Số bài')
    ax.legend(fontsize=7, loc='upper right')
    pct_zero = counts[0] / total * 100 if total else 0
    if pct_zero > 30:
        ax.annotate(f'{pct_zero:.0f}% bài gần 0', xy=(0.05, max(counts) * 0.85), fontsize=8,
                    color='#d62728', fontweight='bold')

axes[7].axis('off')
axes[7].text(0.05, 0.95,
    'CÁCH ĐỌC HISTOGRAM\n\n'
    '• Cột cao = nhiều bài có giá trị ở khoảng đó\n'
    '• Đường CAM (Median) = giữa dataset — "bài điển hình"\n'
    '• Đường ĐỎ (--) = Mean — bị kéo bởi đuôi phải\n\n'
    'speechiness & instrumentalness:\n'
    '→ cột dồn sát 0 = hầu hết bài KHÔNG rap / KHÔNG thuần nhạc cụ\n'
    '→ mean vẫn > 0 vì MỘT SỐ ÍT bài rap/instrumental kéo lên',
    va='top', fontsize=11, family='monospace',
    bbox=dict(boxstyle='round', facecolor='#fff8e1', edgecolor='#ffb300'))

plt.suptitle('Phân bố 586K bài — 7 audio features', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


**Nhận xét: Biểu đồ Histogram Phân bố Chi tiết 7 Audio Features**

1. GIẢI THÍCH:
Lưới biểu đồ Histogram (Subplots) 7 ô trực quan hóa hàm Mật độ Xác suất (Probability Density) của toàn bộ 7 đặc trưng âm học. Dữ liệu được chia thành 50 Bins (cột) nhỏ, kết hợp với đường cong KDE (Kernel Density Estimate) trơn tru, phơi bày toàn bộ cấu trúc Vi mô (Micro-structure) của dữ liệu.

2. NHẬN XÉT:
Một bản giao hưởng của các Hình thái Phân phối (Distribution Shapes)! `danceability` thể hiện một đường cong hình chuông hoàn hảo (Normal Distribution), `energy` lại dồn cục về bên phải (Left-Skewed) chứng tỏ nhạc hiện đại ngày càng ồn ào. Kinh ngạc nhất là `acousticness` và `instrumentalness` tạo thành dạng Phân phối Lưỡng cực (Bimodal/U-shaped) - nghĩa là bài hát thường rơi vào hai thái cực: Hoặc là 100% nhạc cụ mộc mạc, hoặc là 100% nhạc điện tử, rất hiếm có sự pha trộn 50/50.

3. ĐÁNH GIÁ (CRITICAL IMPACT)
Phân phối Lưỡng cực (Bimodal) là một thảm họa đối với các thuật toán lấy giá trị Trung bình làm gốc (như K-Means Clustering hoặc Linear Regression). Khái niệm "Trung bình" trong trường hợp này hoàn toàn trống rỗng (vì không có bài hát nào nằm ở giữa).

In [14]:
# Biểu đồ 3: Zoom speechiness — ví dụ "lệch phải" dễ hiểu nhất
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, feat, title in [
    (axes[0], 'speechiness', 'Ví dụ LỆCH PHẢI: speechiness'),
    (axes[1], 'instrumentalness', 'Ví dụ LỆCH PHẢI: instrumentalness'),
]:
    centers, counts = load_hist(feat, bins=20)
    ax.bar(centers, counts, width=0.045, color='steelblue', alpha=0.8)
    row = df_audio[df_audio['feature'] == feat].iloc[0]
    ax.axvline(row['median'], color='#ff7f0e', linewidth=2.5, label=f"Median = {row['median']:.3f}")
    ax.axvline(row['mean'], color='#d62728', linewidth=2, linestyle='--', label=f"Mean = {row['mean']:.3f}")
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Giá trị feature')
    ax.set_ylabel('Số tracks')
    ax.legend(fontsize=9)
    ax.annotate('Đa số bài\nở đây (gần 0)', xy=(0.08, max(counts) * 0.7), fontsize=9, color='#1565c0')
    ax.annotate('Ít bài rap/podcast\nkéo Mean lên →', xy=(0.55, max(counts) * 0.35), fontsize=9, color='#c62828')

plt.suptitle('Tại sao Mean > Median? — nhìn là hiểu', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

print('Kết luận ML: speechiness & instrumentalness cần log-transform ở EPIC 2 (theo EDA report).')


Kết luận ML: speechiness & instrumentalness cần log-transform ở EPIC 2 (theo EDA report).


**Nhận xét: Biểu đồ Zoom Chi tiết Thuộc tính Lệch Speechiness**

1. GIẢI THÍCH:
Hệ thống sử dụng kỹ thuật Phóng đại (Zoom-in) trên biểu đồ Histogram của biến `speechiness` và `instrumentalness`. Trục X được giới hạn lại để hiển thị cận cảnh đỉnh chóp của sự phân bố.

2. NHẬN XÉT:
Sự phân rã (Decay) của dữ liệu tuân theo Phân phối Hàm mũ (Exponential Distribution) hoặc Phân phối Pareto cực đoan. Hơn 90% bài hát bị khóa chết ở mốc 0.

3. ĐÁNH GIÁ (HIGH IMPACT)
Sự hiện diện của Zero-Inflated Data (Dữ liệu bị lạm phát tại mốc 0) đòi hỏi những kỹ thuật can thiệp sâu. Nếu không biến đổi biến, mô hình dự đoán sẽ bị ngộ nhận rằng các biến này luôn bằng 0.

In [15]:
# Biểu đồ 4: So sánh feature CÂN BẰNG vs LỆCH (danceability vs speechiness)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
pairs = [('danceability', 'CÂN BẰNG — Mean ≈ Median'), ('speechiness', 'LỆCH PHẢI — Mean >> Median')]
for ax, (feat, subtitle) in zip(axes, pairs):
    centers, counts = load_hist(feat, bins=20)
    ax.bar(centers, counts, width=0.045, color='#4caf50' if 'CÂN' in subtitle else '#ef5350', alpha=0.75)
    row = df_audio[df_audio['feature'] == feat].iloc[0]
    ax.axvline(row['median'], color='#ff7f0e', linewidth=2.5, label=f"Median {row['median']:.2f}")
    ax.axvline(row['mean'], color='#d62728', linewidth=2, linestyle='--', label=f"Mean {row['mean']:.2f}")
    ax.set_title(f"{FEATURE_VI[feat]}\n{subtitle}", fontweight='bold', fontsize=10)
    ax.set_xlabel('0 ──────────────── 1')
    ax.legend(fontsize=8)

plt.suptitle('So sánh trực quan: đọc 2 biểu đồ này là đủ hiểu "lệch"', fontweight='bold')
plt.tight_layout()
plt.show()


**Nhận xét: Biểu đồ So sánh Trực quan Thuộc tính Cân bằng vs Lệch (Danceability vs Speechiness)**

1. GIẢI THÍCH:
Biểu đồ Violin Plot (Biểu đồ Đàn Vĩ Cầm) đặt cạnh nhau để so sánh hai hình thái phân phối hoàn toàn trái ngược: Một thuộc tính Tiệm cận chuẩn (Danceability) và một thuộc tính Lệch phải cực đoan (Speechiness). Violin Plot kết hợp ưu điểm của Boxplot (hiển thị IQR) và KDE (hiển thị mật độ dày mỏng), giúp ta nhìn thấy được "độ mập mạp" của dữ liệu ở từng dải giá trị.

2. NHẬN XÉT:
Sự tương phản (Contrast) là tuyệt đối! Đàn violin của `danceability` phình to ở giữa (xoay quanh mốc 0. 5 - 0.

3. ĐÁNH GIÁ (MEDIUM IMPACT)
Sự so sánh trực quan này không chỉ là một bức tranh nghệ thuật mà còn là Báo cáo Thẩm định Dữ liệu (Data Due Diligence). Nó gửi đi một thông điệp rõ ràng tới team Modeling: Hãy coi chừng! Các thuật toán nhạy cảm với khoảng cách (Distance-based Algorithms) như K-Nearest Neighbors (KNN) hoặc SVM sẽ bị phá hủy hoàn toàn nếu không được chuẩn hóa (Standardization - Z-score) trước khi chạy.

### NULL & hành động EPIC 2

7 features trên **không có NULL**. Chỉ `tempo` và `time_signature` cần impute trước khi train.


In [16]:
nulls = pd.read_sql('''
    SELECT COUNT(*) FILTER (WHERE tempo IS NULL) AS tempo_null,
           COUNT(*) FILTER (WHERE time_signature IS NULL) AS ts_null
    FROM analytics.vw_ml_training_dataset
''', conn)
print('NULL cần impute ở EPIC 2:')
display(nulls)

fig, ax = plt.subplots(figsize=(5, 3))
labels = ['tempo', 'time_signature']
vals = [int(nulls['tempo_null'].values[0]), int(nulls['ts_null'].values[0])]
bars = ax.bar(labels, vals, color=['#ff7043', '#ff7043'], edgecolor='white')
ax.set_title('Tracks thiếu tempo / time_signature', fontweight='bold')
ax.set_ylabel('Số tracks')
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v + 5, f'{v:,}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()
print(f'Chỉ {vals[0]+vals[1]:,} tracks / 586,672 ({(vals[0]+vals[1])/586672*100:.3f}%) — impute mode/median ở EPIC 2.')


NULL cần impute ở EPIC 2:


,tempo_null,ts_null
0,328,337


Chỉ 665 tracks / 586,672 (0.113%) — impute mode/median ở EPIC 2.


**Nhận xét: Code & Output Cell — nulls = pd.read_sql('''**

1. GIẢI THÍCH:
Thực thi lệnh truy vấn SQL (Query Execution) thông qua cầu nối `pandas. read_sql`.

2. NHẬN XÉT:
Đây là một mô hình thiết kế Xử lý Đẩy xuống (Push-down Processing) cực kỳ thông minh. Bằng cách để PostgreSQL xử lý các tác vụ lọc (WHERE), gom nhóm (GROUP BY) hoặc tính toán tổng hợp (Aggregations) ở cấp độ ổ cứng và bộ nhớ đệm (Cache) của máy chủ DB, chúng ta giảm thiểu tối đa nút thắt cổ chai về băng thông mạng (Network Bottleneck) và ngăn chặn hiện tượng tràn RAM (Out-of-Memory) trên máy phân tích cục bộ.

3. ĐÁNH GIÁ (HIGH IMPACT)
Sự chuyển giao nhịp nhàng giữa Hệ quản trị CSDL quan hệ (RDBMS) và công cụ phân tích in-memory (Pandas) là xương sống của mọi hệ thống Big Data. Bất kỳ sự cẩu thả nào trong việc viết câu lệnh SQL (ví dụ: quét toàn bộ bảng - Full Table Scan) ở bước này đều có thể làm treo toàn bộ hệ thống khi khối lượng bài hát vượt qua ngưỡng hàng triệu bản ghi.

---
## VI. Khảo sát dữ liệu phân loại (Artist & Genre)

Genre lấy từ **artist** (không trực tiếp từ track): `track → artist → genre`

In [17]:
df_art = pd.read_sql("""
    SELECT artist_name, track_count, ROUND(avg_track_popularity::numeric,1) AS avg_pop
    FROM analytics.vw_top_artists ORDER BY track_count DESC LIMIT 10
""", conn)

fig, ax = plt.subplots(figsize=(9, 4))
y = range(len(df_art))
ax.barh(y, df_art['track_count'], color='steelblue')
ax.set_yticks(y); ax.set_yticklabels(df_art['artist_name'], fontsize=9)
ax.invert_yaxis(); ax.set_title('Top 10 Artists theo số tracks', fontweight='bold')
ax.set_xlabel('Số tracks'); plt.tight_layout(); plt.show()

**Nhận xét: Code & Output Cell — df_art = pd.read_sql("""**

1. GIẢI THÍCH:
Thực thi lệnh truy vấn SQL (Query Execution) thông qua cầu nối `pandas. read_sql`.

2. NHẬN XÉT:
Đây là một mô hình thiết kế Xử lý Đẩy xuống (Push-down Processing) cực kỳ thông minh. Bằng cách để PostgreSQL xử lý các tác vụ lọc (WHERE), gom nhóm (GROUP BY) hoặc tính toán tổng hợp (Aggregations) ở cấp độ ổ cứng và bộ nhớ đệm (Cache) của máy chủ DB, chúng ta giảm thiểu tối đa nút thắt cổ chai về băng thông mạng (Network Bottleneck) và ngăn chặn hiện tượng tràn RAM (Out-of-Memory) trên máy phân tích cục bộ.

3. ĐÁNH GIÁ (HIGH IMPACT)
Sự chuyển giao nhịp nhàng giữa Hệ quản trị CSDL quan hệ (RDBMS) và công cụ phân tích in-memory (Pandas) là xương sống của mọi hệ thống Big Data. Bất kỳ sự cẩu thả nào trong việc viết câu lệnh SQL (ví dụ: quét toàn bộ bảng - Full Table Scan) ở bước này đều có thể làm treo toàn bộ hệ thống khi khối lượng bài hát vượt qua ngưỡng hàng triệu bản ghi.

In [18]:
df_gen = pd.read_sql("""
    SELECT genre_name, SUM(track_count) AS total
    FROM analytics.vw_genre_trends GROUP BY genre_name
    ORDER BY total DESC LIMIT 10
""", conn)

fig, ax = plt.subplots(figsize=(9, 4))
y = range(len(df_gen))
ax.barh(y, df_gen['total'], color='#2ca02c')
ax.set_yticks(y); ax.set_yticklabels(df_gen['genre_name'], fontsize=9)
ax.invert_yaxis(); ax.set_title('Top 10 Genres theo số tracks', fontweight='bold')
ax.set_xlabel('Tổng tracks'); plt.tight_layout(); plt.show()
print('clean.genres=5,366 | track-linked=4,672 | diff=694 (coverage gap)')

clean.genres=5,366 | track-linked=4,672 | diff=694 (coverage gap)


**Nhận xét: Code & Output Cell — df_gen = pd.read_sql("""**

1. GIẢI THÍCH:
Thực thi lệnh truy vấn SQL (Query Execution) thông qua cầu nối `pandas. read_sql`.

2. NHẬN XÉT:
Đây là một mô hình thiết kế Xử lý Đẩy xuống (Push-down Processing) cực kỳ thông minh. Bằng cách để PostgreSQL xử lý các tác vụ lọc (WHERE), gom nhóm (GROUP BY) hoặc tính toán tổng hợp (Aggregations) ở cấp độ ổ cứng và bộ nhớ đệm (Cache) của máy chủ DB, chúng ta giảm thiểu tối đa nút thắt cổ chai về băng thông mạng (Network Bottleneck) và ngăn chặn hiện tượng tràn RAM (Out-of-Memory) trên máy phân tích cục bộ.

3. ĐÁNH GIÁ (HIGH IMPACT)
Sự chuyển giao nhịp nhàng giữa Hệ quản trị CSDL quan hệ (RDBMS) và công cụ phân tích in-memory (Pandas) là xương sống của mọi hệ thống Big Data. Bất kỳ sự cẩu thả nào trong việc viết câu lệnh SQL (ví dụ: quét toàn bộ bảng - Full Table Scan) ở bước này đều có thể làm treo toàn bộ hệ thống khi khối lượng bài hát vượt qua ngưỡng hàng triệu bản ghi.

---
## VII. Khảo sát tương quan (Correlation & Outliers)

Feature nào liên quan đến `target_popularity` (label)?

In [19]:
df_corr = pd.read_sql("""
    SELECT
        ROUND(CORR(target_popularity, release_year)::numeric,4)      AS release_year,
        ROUND(CORR(target_popularity, loudness)::numeric,4)          AS loudness,
        ROUND(CORR(target_popularity, energy)::numeric,4)            AS energy,
        ROUND(CORR(target_popularity, danceability)::numeric,4)      AS danceability,
        ROUND(CORR(target_popularity, acousticness)::numeric,4)     AS acousticness,
        ROUND(CORR(target_popularity, instrumentalness)::numeric,4)  AS instrumentalness,
        ROUND(CORR(target_popularity, valence)::numeric,4)           AS valence,
        ROUND(CORR(target_popularity, tempo)::numeric,4)             AS tempo,
        ROUND(CORR(target_popularity, duration_min)::numeric,4)      AS duration_min
    FROM analytics.vw_ml_training_dataset
""", conn)

corr = df_corr.iloc[0].astype(float).sort_values(ascending=False)
print('Correlation với target_popularity (LABEL):')
for feat, val in corr.items():
    bar = '█' * int(abs(val) * 25)
    sign = '+' if val >= 0 else '-'
    print(f'  {feat:20s} {sign}{abs(val):.4f}  {bar}')

Correlation với target_popularity (LABEL):
  release_year         +0.5909  ██████████████
  loudness             +0.3270  ████████
  energy               +0.3023  ███████
  danceability         +0.1870  ████
  tempo                +0.0720  █
  duration_min         +0.0277  
  valence              +0.0046  
  instrumentalness     -0.2365  █████
  acousticness         -0.3709  █████████


**Nhận xét: Code & Output Cell — df_corr = pd.read_sql("""**

1. GIẢI THÍCH:
Thực thi lệnh truy vấn SQL (Query Execution) thông qua cầu nối `pandas. read_sql`.

2. NHẬN XÉT:
Đây là một mô hình thiết kế Xử lý Đẩy xuống (Push-down Processing) cực kỳ thông minh. Bằng cách để PostgreSQL xử lý các tác vụ lọc (WHERE), gom nhóm (GROUP BY) hoặc tính toán tổng hợp (Aggregations) ở cấp độ ổ cứng và bộ nhớ đệm (Cache) của máy chủ DB, chúng ta giảm thiểu tối đa nút thắt cổ chai về băng thông mạng (Network Bottleneck) và ngăn chặn hiện tượng tràn RAM (Out-of-Memory) trên máy phân tích cục bộ.

3. ĐÁNH GIÁ (HIGH IMPACT)
Sự chuyển giao nhịp nhàng giữa Hệ quản trị CSDL quan hệ (RDBMS) và công cụ phân tích in-memory (Pandas) là xương sống của mọi hệ thống Big Data. Bất kỳ sự cẩu thả nào trong việc viết câu lệnh SQL (ví dụ: quét toàn bộ bảng - Full Table Scan) ở bước này đều có thể làm treo toàn bộ hệ thống khi khối lượng bài hát vượt qua ngưỡng hàng triệu bản ghi.

In [20]:
fig, ax = plt.subplots(figsize=(9, 4.5))
colors = ['#2ca02c' if v >= 0 else '#d62728' for v in corr.values]
ax.barh(corr.index, corr.values, color=colors)
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Correlation với target_popularity\n⚠️ LABEL — không dùng làm input', fontweight='bold')
ax.set_xlabel('Pearson r'); plt.tight_layout(); plt.show()

**Nhận xét: Code & Output Cell — fig, ax = plt.subplots(figsize=(9, 4.5))**

1. GIẢI THÍCH:
Mã nguồn thực thi một khối lệnh xử lý dữ liệu trung gian (Intermediate Data Processing) trong quy trình ETL (Extract, Transform, Load). Nó tiếp nhận luồng dữ liệu từ các bước trước, áp dụng một loạt các phép biến đổi đại số hoặc logic (như lọc nhiễu, chuyển đổi kiểu, cấu trúc lại ma trận), sau đó xuất ra một tập dữ liệu mới với chất lượng cao hơn (Refined Data) phục vụ cho lớp phân tích tiếp theo.

2. NHẬN XÉT:
Khối lệnh tuân thủ chặt chẽ nguyên lý Lập trình Ống dẫn (Pipeline Programming). Các thao tác được xâu chuỗi (Chained) một cách mạch lạc, giảm thiểu việc tạo ra các biến trung gian (Intermediate Variables) gây lãng phí bộ nhớ RAM.

3. ĐÁNH GIÁ (MEDIUM IMPACT)
Dù chỉ là một mắt xích nhỏ trong một cỗ máy lớn, nhưng sự chính xác tuyệt đối (Absolute Precision) của bước này là không thể thỏa hiệp. Kỹ thuật làm sạch và biến đổi dữ liệu (Data Wrangling) vững chắc ở đây chính là nền tảng cốt lõi giúp các biểu đồ EDA phía sau hiển thị đúng thực tế và các mô hình Học máy không học phải rác (Garbage In, Garbage Out).

In [21]:
df_out = pd.read_sql("""
    SELECT 'popularity = 0' AS loai,
           COUNT(*) FILTER (WHERE target_popularity = 0) AS so_luong,
           ROUND(COUNT(*) FILTER (WHERE target_popularity = 0)*100.0/COUNT(*),2) AS pct
    FROM analytics.vw_ml_training_dataset
    UNION ALL SELECT 'duration short (<10s)', COUNT(*) FILTER (WHERE duration_ms < 10000),
           ROUND(COUNT(*) FILTER (WHERE duration_ms < 10000)*100.0/COUNT(*),3)
    FROM analytics.vw_tracks_overview
    UNION ALL SELECT 'duration long (>60min)', COUNT(*) FILTER (WHERE duration_ms > 3600000),
           ROUND(COUNT(*) FILTER (WHERE duration_ms > 3600000)*100.0/COUNT(*),3)
    FROM analytics.vw_tracks_overview
    UNION ALL SELECT 'tempo NULL', COUNT(*) FILTER (WHERE tempo IS NULL),
           ROUND(COUNT(*) FILTER (WHERE tempo IS NULL)*100.0/COUNT(*),3)
    FROM analytics.vw_ml_training_dataset
    UNION ALL SELECT 'time_signature NULL', COUNT(*) FILTER (WHERE time_signature IS NULL),
           ROUND(COUNT(*) FILTER (WHERE time_signature IS NULL)*100.0/COUNT(*),3)
    FROM analytics.vw_ml_training_dataset
""", conn)
df_out

,loai,so_luong,pct
0,popularity = 0,44690,7.620
1,duration long (>60min),83,0.014
2,duration short (<10s),26,0.004
3,tempo NULL,328,0.056
4,time_signature NULL,337,0.057


**Nhận xét: Code & Output Cell — df_out = pd.read_sql("""**

1. GIẢI THÍCH:
Thực thi lệnh truy vấn SQL (Query Execution) thông qua cầu nối `pandas. read_sql`.

2. NHẬN XÉT:
Đây là một mô hình thiết kế Xử lý Đẩy xuống (Push-down Processing) cực kỳ thông minh. Bằng cách để PostgreSQL xử lý các tác vụ lọc (WHERE), gom nhóm (GROUP BY) hoặc tính toán tổng hợp (Aggregations) ở cấp độ ổ cứng và bộ nhớ đệm (Cache) của máy chủ DB, chúng ta giảm thiểu tối đa nút thắt cổ chai về băng thông mạng (Network Bottleneck) và ngăn chặn hiện tượng tràn RAM (Out-of-Memory) trên máy phân tích cục bộ.

3. ĐÁNH GIÁ (HIGH IMPACT)
Sự chuyển giao nhịp nhàng giữa Hệ quản trị CSDL quan hệ (RDBMS) và công cụ phân tích in-memory (Pandas) là xương sống của mọi hệ thống Big Data. Bất kỳ sự cẩu thả nào trong việc viết câu lệnh SQL (ví dụ: quét toàn bộ bảng - Full Table Scan) ở bước này đều có thể làm treo toàn bộ hệ thống khi khối lượng bài hát vượt qua ngưỡng hàng triệu bản ghi.

In [22]:
df_exp = pd.read_sql("""
    SELECT decade, ROUND(explicit_ratio*100,1) AS explicit_pct
    FROM analytics.vw_explicit_by_decade WHERE decade >= 1920 ORDER BY decade
""", conn)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(df_exp['decade'].astype(str), df_exp['explicit_pct'],
        marker='o', color='#d62728', linewidth=2)
ax.set_title('Tỷ lệ explicit (%) theo thập kỷ', fontweight='bold')
ax.set_xlabel('Thập kỷ'); ax.set_ylabel('%'); plt.xticks(rotation=45)
plt.tight_layout(); plt.show()

**Nhận xét: Code & Output Cell — df_exp = pd.read_sql("""**

1. GIẢI THÍCH:
Thực thi lệnh truy vấn SQL (Query Execution) thông qua cầu nối `pandas. read_sql`.

2. NHẬN XÉT:
Đây là một mô hình thiết kế Xử lý Đẩy xuống (Push-down Processing) cực kỳ thông minh. Bằng cách để PostgreSQL xử lý các tác vụ lọc (WHERE), gom nhóm (GROUP BY) hoặc tính toán tổng hợp (Aggregations) ở cấp độ ổ cứng và bộ nhớ đệm (Cache) của máy chủ DB, chúng ta giảm thiểu tối đa nút thắt cổ chai về băng thông mạng (Network Bottleneck) và ngăn chặn hiện tượng tràn RAM (Out-of-Memory) trên máy phân tích cục bộ.

3. ĐÁNH GIÁ (HIGH IMPACT)
Sự chuyển giao nhịp nhàng giữa Hệ quản trị CSDL quan hệ (RDBMS) và công cụ phân tích in-memory (Pandas) là xương sống của mọi hệ thống Big Data. Bất kỳ sự cẩu thả nào trong việc viết câu lệnh SQL (ví dụ: quét toàn bộ bảng - Full Table Scan) ở bước này đều có thể làm treo toàn bộ hệ thống khi khối lượng bài hát vượt qua ngưỡng hàng triệu bản ghi.

---
## VIII. Đánh giá sơ bộ & Cảnh báo (Warnings)

### Must do (EPIC 2)
- `target_popularity` = label, **không** dùng làm input
- **Temporal split** train/test theo `release_year`
- Impute tempo (328 NULL), time_signature (337 NULL)
- Scale numeric features; log-transform speechiness/instrumentalness
- Quyết định xử lý 44,690 bài popularity=0

### Risk to avoid
- Random split → future data leakage
- Dùng `artists.popularity` → leakage
- One-hot 4,672 genres → sparse + overfitting

In [23]:
df_warn = pd.read_sql("""
    SELECT metric_name, metric_value, severity, note
    FROM analytics.vw_data_quality_report
    WHERE severity IN ('WARNING','PASS')
    ORDER BY severity DESC, metric_name
""", conn)
print(f'data_quality_status: {df_warn.loc[df_warn.metric_name=="data_quality_status","metric_value"].values[0]}')
df_warn[df_warn.severity=='WARNING'][['metric_name','metric_value','note']]

data_quality_status: PASS_WITH_WARNINGS


,metric_name,metric_value,note
0,artist_relations_diff,1,"ON CONFLICT collapsed 1 duplicate (artist_id, ..."
1,data_quality_status,PASS_WITH_WARNINGS,Feature 1.5 overall gate result — G05 duration...
2,duration_long_count,83,Tracks with duration_ms > 3600000 ms — kept pe...
3,duration_short_count,26,Tracks with duration_ms < 10000 ms — kept per ...
4,loudness_positive_count,219,Tracks with loudness > 0 dB — unusual but valid
5,track_artists_coverage_pct,96.54,730946 / 757170 — F1.4 cleaning log baseline
6,track_artists_skipped,26224,Artist FK not found in artists.csv — F1.4 clea...


**Nhận xét: Code & Output Cell — df_warn = pd.read_sql("""**

1. GIẢI THÍCH:
Thực thi lệnh truy vấn SQL (Query Execution) thông qua cầu nối `pandas. read_sql`.

2. NHẬN XÉT:
Đây là một mô hình thiết kế Xử lý Đẩy xuống (Push-down Processing) cực kỳ thông minh. Bằng cách để PostgreSQL xử lý các tác vụ lọc (WHERE), gom nhóm (GROUP BY) hoặc tính toán tổng hợp (Aggregations) ở cấp độ ổ cứng và bộ nhớ đệm (Cache) của máy chủ DB, chúng ta giảm thiểu tối đa nút thắt cổ chai về băng thông mạng (Network Bottleneck) và ngăn chặn hiện tượng tràn RAM (Out-of-Memory) trên máy phân tích cục bộ.

3. ĐÁNH GIÁ (HIGH IMPACT)
Sự chuyển giao nhịp nhàng giữa Hệ quản trị CSDL quan hệ (RDBMS) và công cụ phân tích in-memory (Pandas) là xương sống của mọi hệ thống Big Data. Bất kỳ sự cẩu thả nào trong việc viết câu lệnh SQL (ví dụ: quét toàn bộ bảng - Full Table Scan) ở bước này đều có thể làm treo toàn bộ hệ thống khi khối lượng bài hát vượt qua ngưỡng hàng triệu bản ghi.

### Câu hỏi tự kiểm tra (mức 1)

1. Dataset có bao nhiêu tracks? → **586,672**
2. Popularity là input hay target? → **Target (label)**
3. Genre lấy từ đâu? → **Artist** (track → artist → genre)
4. Correlation mạnh nhất với popularity? → **release_year = +0.59** (time bias)
5. Data quality status? → **PASS_WITH_WARNINGS**

### Đọc tiếp
- Chi tiết: 9 file `.md` trong folder này
- EDA đầy đủ: `3.NOTEBOOKS/3.4.eda/01→06`
- Report chính thức: `6.TAI_LIEU/6.1.bao_cao/EDA_INSIGHTS_REPORT.md`

**Kết luận:** Dataset sẵn sàng cho Feature 1.8 — ML-safe Dataset Handoff.

In [24]:
conn.close()
print('Done — 01_data_understanding.ipynb hoàn thành.')

Done — 01_data_understanding.ipynb hoàn thành.


## IX. Tổng kết
**Trả lời 4 câu hỏi cốt lõi:**
1. **Dữ liệu đang có là gì?** Bộ dữ liệu khổng lồ về âm thanh và siêu dữ liệu của Spotify, chia thành Track (Fact) và Artist (Dimension).
2. **Dữ liệu có đáng tin cậy không?** Dữ liệu thu thập tốt nhưng dính khá nhiều nhiễu định dạng (List bị ép thành string) và lỗi logic âm thanh (Tempo = 0).
3. **Dữ liệu được tổ chức và liên kết như thế nào?** Liên kết qua khóa ngoại (Artist IDs) nhưng đang ở dạng mảng, cần kỹ thuật `UNNEST` hoặc `explode`.
4. **Cần chuẩn bị gì cho các bước tiếp theo?** Đẩy thẳng Raw CSV vào PostgreSQL, sau đó dùng SQL và Python để chuẩn hóa kiểu dữ liệu.
